In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

In [ ]:
# Enable interactive plot
%matplotlib notebook

## Add Path

In [ ]:
import os
import sys

In [ ]:
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
del module_path

## Organize imports

In [ ]:
from typing import Dict, List, Optional, Tuple, Union

In [ ]:
from collections import OrderedDict

In [ ]:
from pathlib import Path

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
import random

In [ ]:
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet18, resnet34, resnet50
from torchvision import transforms
from torchvision.datasets import FashionMNIST, CIFAR10, STL10
import time
from tqdm.autonotebook import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import (precision_score, 
    recall_score, f1_score, accuracy_score)

In [ ]:
import kornia.augmentation as K

In [ ]:
from kornia.augmentation.utils import _shape_validation
from kornia.geometry.bbox import bbox_to_mask, infer_bbox_shape
from kornia.constants import BorderType, Resample
from kornia.augmentation import random_generator as rg

In [ ]:
import pytorch_lightning as pl

In [ ]:
torch.manual_seed(1)

## Define funcrtions

Define functions for model approximation

#### Function space

In [ ]:
def funct_space(xs: np.ndarray, func: callable) -> np.ndarray:
    return func(xs)
    

# Sample X space

Generate random sample from $X$ spaces

In [ ]:
X = np.random.uniform(low=-1, high=1, size=(200,))
X.shape, X

In [ ]:
y = funct_space(X, lambda x: x**2)
y.shape, y

In [ ]:
# list(zip(X, y))

In [ ]:
plt.plot(X, y, 'bo')
plt.show()
# plt.close()

## Define model

In [ ]:
in_features = 1
out_features = 1
hidden_features = 10

In [ ]:
class LinNN(nn.Module):
    
    def __init__(self, n_in: int = 1, n_midd: int=4, n_out: int = 1):
        super().__init__()
        self.fc1 = nn.Linear(n_in, n_midd, bias=True)
        self.fc2 = nn.Linear(n_midd, n_out, bias=True)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc1(x)
        z = self.fc2(h)
        
        return z

In [ ]:
def init_net(
    in_features=in_features, hidden_features=hidden_features, 
    out_features=out_features):
    return nn.Sequential(OrderedDict([
            ('fn1', nn.Linear(in_features, hidden_features, bias=True)),
            ('a1', nn.ReLU()),
            ('fn2', nn.Linear(hidden_features, out_features, bias=True)),
        ]))

## Set-up the training

In [ ]:
func_ds = TensorDataset(torch.Tensor(X),torch.Tensor(y)) # create your datset
func_dl = DataLoader(func_ds)

In [ ]:
net = init_net()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)
loss = torch.nn.MSELoss()

In [ ]:
class LtModel(pl.LightningModule):
    def __init__(self, net: nn.Module, opt, loss):
        super().__init__()
        self.net = net
        self.opt = opt
        self.loss = loss
        self.imgs = list()
        fig, ax = self._init_plt()
        self.fig = fig
        self.ax = ax
        
    @staticmethod
    def _init_plt():
        plt.figure(figsize=(10,4))
        plt.scatter(X, y, color = "orange")
        plt.title('Regression Analysis')
        plt.xlabel('Independent varible')
        plt.ylabel('Dependent varible')
        plt.show()
        fig, ax = plt.subplots(figsize=(12,7))
        
        return fig, ax
    
    @staticmethod
    def plot_progress(ax, fig, x_t, y_t, y_hat, loss, t):
        # plot and show learning process
        x_v = x_t.to('cpu').data.numpy()
        y_v = y_t.to('cpu').data.numpy()
        y_hat_v = y_hat.to('cpu').data.numpy()
        ls_v = loss.to('cpu').data.numpy()
        plt.cla()
        ax.set_title('Regression Analysis', fontsize=35)
        ax.set_xlabel('Independent variable', fontsize=24)
        ax.set_ylabel('Dependent variable', fontsize=24)
        ax.set_xlim(-1.0, 1.0)
        ax.set_ylim(-1.0, 1.0)
        ax.scatter(x_v, y_v, color = "orange")
        #         ax.plot(x_v, y_hat_v, 'g-', lw=3)
        ax.scatter(x_v, y_hat_v, color = "blue")
        ax.text(1.0, 0.1, 'Step = %d' % t, fontdict={'size': 24, 'color':  'red'})
        ax.text(1.0, 0, 'Loss = %.4f' % ls_v,
                fontdict={'size': 24, 'color':  'red'})

        # Used to return the plot as an image array 
        # (https://ndres.me/post/matplotlib-animated-gifs-easily/)
        fig.canvas.draw()       # draw the canvas, cache the renderer
        
        image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
        image  = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        
        return image

    def forward(self, x):
        return self.net(x)
    
    def training_step(self, batch, batch_idx):
        # training_step defined the train loop.
        # It is independent of forward
        x, y = batch
        y_hat = self(x)
        loss = self.loss(y_hat, y)
        # Logging to TensorBoard by default
        self.log("train_loss", loss)
        img = self.plot_progress(self,ax, self.fig, x, y, y_hat, loss, batch_idx)
        self.imgs.append(img)
        
        return loss

    def configure_optimizers(self):
        return self.opt

## Perform custom training

In [ ]:
X_t = torch.unsqueeze(torch.Tensor(X), dim=1)
y_t = torch.unsqueeze(torch.Tensor(y), dim=1)
X_t.shape, X_t.dtype

In [ ]:
plt.close()
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
loss = torch.nn.MSELoss()
fig, ax = LtModel._init_plt()
net = init_net()
epochs  = 500
net.train()
imgs = list()
for ep in range(epochs):
    y_hat = net(X_t)
    y_hat = F.sigmoid(y_hat)
    ls = loss(y_hat, y_t)
    optimizer.zero_grad()
    ls.backward()
    optimizer.step()
    img = LtModel.plot_progress(ax, fig, X_t, y_t, y_hat, ls, ep + 1)
    imgs.append(img)

In [ ]:
fig, ax = plt.subplots() 
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)

scatter, = ax.plot(
    [], [], 'go', label='Initiali function walues', color='orange')
line, = ax.plot([], [], 'r', label='Approximation values')
ax.legend()

def anim(frame_number):
    y_hat = net(X_t)
    y_hat = F.sigmoid(y_hat)
    ls = loss(y_hat, y_t)
    optimizer.zero_grad()
    ls.backward()
    optimizer.step()
    
    y_hat_np = torch.squeeze(y_hat).to('cpu').data.numpy()
    scatter.set_data((X, y))
    line.set_data((X, y_hat_np))
    

optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
loss = torch.nn.MSELoss()
fig, ax = LtModel._init_plt()
net = init_net()
epochs  = 500
net.train()
anm = FuncAnimation(fig, anim, frames=500, interval=5)

In [ ]:
import imageio

In [ ]:
imageio.mimsave('./curve_1.gif', imgs, fps=10)

## Perform training

In [ ]:
lt_model = LtModel(net, optimizer, loss)

In [ ]:
trainer = pl.Trainer(max_epochs=10)
trainer.fit(lt_model, func_dl)